# Match Selection for YOLO Annotation Diversity

Goal: Select ~10 new matches to annotate (60 frames each → ~600 new frames) to diversify the training set.

**Already annotated (from video folder):**
1. Arsenal – Dečić (21.03.2026) — Stadium: Tivat
2. Bokelj – Jedinstvo (01.03.2026, R22) — Stadium: Kotor
3. Budućnost – Sutjeska (21.02.2026, R20) — Stadium: Podgorica
4. Jezero – Jedinstvo (21.03.2026, R26) — Stadium: Berane
5. Petrovac – Mornar (21.03.2026, R26) — Stadium: Petrovac
6. Sutjeska – Mladost (unclear date) — Stadium: Nikšić

**Criteria for new matches:**
- Cover all 10 stadiums (esp. uncovered: Bar, Bijelo Polje, Tuzi, Danilovgrad)
- Every team appears at least once
- Mix of early/late kickoff times (day vs dusk/night)
- Different rounds/months for lighting variety

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('data/processed/cg')
df = pd.read_csv(DATA_DIR / 'matches_metadata.csv')
df['match_datetime'] = pd.to_datetime(df['match_datetime'])
df['match_date'] = pd.to_datetime(df['match_date'])
df['hour'] = df['match_datetime'].dt.hour
df['month'] = df['match_datetime'].dt.month
print(f'Total matches: {len(df)}')
df.head()

Total matches: 130


,match_id,round,match_date,match_datetime,status,homeTeam_id,homeTeam_name,homeTeam_formation,awayTeam_id,awayTeam_name,awayTeam_formation,homeScore_period1,homeScore_period2,homeScore_total,awayScore_period1,awayScore_period2,awayScore_total,hour,month
0,14147310,1,2025-08-04,2025-08-04 18:00:00,Ended,37956,FK Arsenal Tivat,3-4-2-1,6219,FK Jedinstvo Bijelo Polje,4-2-3-1,1.0,1.0,2.0,0.0,0.0,0.0,18,8
1,14147408,1,2025-08-04,2025-08-04 18:00:00,Ended,6216,OFK Petrovac,4-4-2,7581,FK Bokelj Kotor,4-2-3-1,1.0,1.0,2.0,0.0,0.0,0.0,18,8
2,14147300,1,2025-08-04,2025-08-04 18:00:00,Ended,6226,FK Dečić Tuzi,4-1-4-1,327162,Mladost DG,4-2-3-1,2.0,1.0,3.0,0.0,2.0,2.0,18,8
3,14147319,1,2025-08-04,2025-08-04 18:00:00,Ended,24312,FK Jezero,4-4-2,5143,FK Budućnost Podgorica,4-2-3-1,1.0,0.0,1.0,0.0,2.0,2.0,18,8
4,14147314,1,2025-08-04,2025-08-04 19:00:00,Ended,6224,FK Sutjeska Nikšić,4-2-3-1,36019,FK Mornar Bar,4-2-3-1,0.0,3.0,3.0,0.0,0.0,0.0,19,8


In [6]:
# Parse FSCG page for actual stadium names per match
from bs4 import BeautifulSoup
import re

with open('outputs/fscg_page.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

# Find the "SVE UTAKMICE" table (the second table with class "fixtures style2")
tables = soup.find_all('table', class_='fixtures')
print(f"Found {len(tables)} fixture tables")

# The second table has the full "SVE UTAKMICE" data (all rounds)
all_matches_table = tables[1] if len(tables) > 1 else tables[0]

records = []
current_round = None

for row in all_matches_table.find_all('tr'):
    # Check if it's a round header
    h3 = row.find('span', class_='h3')
    if h3:
        round_text = h3.get_text(strip=True)
        match = re.search(r'(\d+)\.\s*kolo', round_text)
        if match:
            current_round = int(match.group(1))
        continue
    
    # Skip if no current round yet
    if current_round is None:
        continue
    
    tds = row.find_all('td')
    if len(tds) < 4:
        continue
    
    # Parse date/time
    date_div = tds[0].find('div', class_='date')
    time_div = tds[0].find('div', class_='time')
    date_str = date_div.get_text(strip=True) if date_div else ''
    time_str = time_div.get_text(strip=True) if time_div else ''
    
    # Stadium
    stadium = tds[1].get_text(strip=True)
    
    # Home team
    home_link = tds[2].find('a')
    home_team = home_link.get_text(strip=True) if home_link else tds[2].get_text(strip=True)
    
    # Away team  
    away_link = tds[3].find('a')
    away_team = away_link.get_text(strip=True) if away_link else tds[3].get_text(strip=True)
    
    # Scores
    home_score = tds[2].find('span', class_='res1')
    away_score = tds[3].find('span', class_='res2')
    
    records.append({
        'round': current_round,
        'date': date_str,
        'time': time_str,
        'stadium': stadium,
        'home_team': home_team,
        'away_team': away_team,
    })

fscg_df = pd.DataFrame(records)
print(f"Total matches parsed: {len(fscg_df)}")
print(f"Rounds covered: {fscg_df['round'].min()} to {fscg_df['round'].max()}")
print(f"\nAll unique stadiums across ALL rounds:")
for i, s in enumerate(sorted(fscg_df['stadium'].unique()), 1):
    count = (fscg_df['stadium'] == s).sum()
    print(f"  {i}. {s} ({count} matches)")

Found 2 fixture tables
Total matches parsed: 180
Rounds covered: 1 to 36

All unique stadiums across ALL rounds:
  1. Arena Besa (18 matches)
  2. DG arena (14 matches)
  3. Gradski stadion (40 matches)
  4. Gradski stadion Berane (15 matches)
  5. Gradski stadion Nikšić (18 matches)
  6. SRC Topolica (18 matches)
  7. Stadion FK Arsenal (18 matches)
  8. Stadion Mitar Mićo Goliš (16 matches)
  9. Stadion pod Vrmcem (18 matches)
  10. Trening kamp FSCG (5 matches)


In [7]:
# Apply domain knowledge corrections to disambiguate "Gradski stadion"
# 1. Jezero home "Gradski stadion" → "Gradski stadion Berane"  (Jezero plays in Berane)
# 2. Jedinstvo home "Gradski stadion" → "Gradski stadion Bijelo Polje"  (except R02 which FSCG already labels "Gradski stadion Berane")
# 3. Budućnost home "Gradski stadion" → "Gradski stadion Podgorica"
# 4. R23 Mladost DG vs Budućnost (07.03.2026) → "Gradski stadion Pljevlja"

# First re-parse from original HTML (undo any previous fix)
with open('outputs/fscg_page.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')
tables = soup.find_all('table', class_='fixtures')
all_matches_table = tables[1] if len(tables) > 1 else tables[0]

records = []
current_round = None
for row in all_matches_table.find_all('tr'):
    h3 = row.find('span', class_='h3')
    if h3:
        round_text = h3.get_text(strip=True)
        m = re.search(r'(\d+)\.\s*kolo', round_text)
        if m:
            current_round = int(m.group(1))
        continue
    if current_round is None:
        continue
    tds = row.find_all('td')
    if len(tds) < 4:
        continue
    date_div = tds[0].find('div', class_='date')
    time_div = tds[0].find('div', class_='time')
    home_link = tds[2].find('a')
    away_link = tds[3].find('a')
    records.append({
        'round': current_round,
        'date': date_div.get_text(strip=True) if date_div else '',
        'time': time_div.get_text(strip=True) if time_div else '',
        'stadium': tds[1].get_text(strip=True),
        'home_team': home_link.get_text(strip=True) if home_link else '',
        'away_team': away_link.get_text(strip=True) if away_link else '',
    })

fscg_df = pd.DataFrame(records)

def fix_stadium(row):
    s = row['stadium']
    if s != 'Gradski stadion':
        return s
    home = row['home_team']
    # Special case: R23 Mladost DG vs Budućnost at Pljevlja
    if row['round'] == 23 and 'Mladost' in home and row['date'] == '07.03.2026.':
        return 'Gradski stadion Pljevlja'
    if home == 'Jezero':
        return 'Gradski stadion Berane'
    if 'Jedinstvo' in home:
        return 'Gradski stadion Bijelo Polje'
    if home == 'Budućnost':
        return 'Gradski stadion Podgorica'
    return s

fscg_df['stadium'] = fscg_df.apply(fix_stadium, axis=1)
fscg_r26 = fscg_df[fscg_df['round'] <= 26].copy()

print("Corrected unique stadiums (rounds 1-26):")
for i, s in enumerate(sorted(fscg_r26['stadium'].unique()), 1):
    count = (fscg_r26['stadium'] == s).sum()
    print(f"  {i}. {s} ({count} matches)")
print(f"\nTotal unique stadiums: {fscg_r26['stadium'].nunique()}")

remaining = fscg_r26[fscg_r26['stadium'] == 'Gradski stadion']
if len(remaining) > 0:
    print(f"\n⚠️ Still {len(remaining)} matches with ambiguous 'Gradski stadion':")
    print(remaining[['round', 'date', 'home_team', 'away_team']].to_string(index=False))
else:
    print("\n✓ No ambiguous 'Gradski stadion' remaining")

Corrected unique stadiums (rounds 1-26):
  1. Arena Besa (13 matches)
  2. DG arena (9 matches)
  3. Gradski stadion Berane (15 matches)
  4. Gradski stadion Bijelo Polje (11 matches)
  5. Gradski stadion Nikšić (14 matches)
  6. Gradski stadion Pljevlja (1 matches)
  7. Gradski stadion Podgorica (12 matches)
  8. SRC Topolica (12 matches)
  9. Stadion FK Arsenal (14 matches)
  10. Stadion Mitar Mićo Goliš (12 matches)
  11. Stadion pod Vrmcem (13 matches)
  12. Trening kamp FSCG (4 matches)

Total unique stadiums: 12

✓ No ambiguous 'Gradski stadion' remaining


In [ ]:
# Filter to rounds 1-26 only and analyze stadium usage
fscg_r26 = fscg_df[fscg_df['round'] <= 26].copy()
print(f"Matches in rounds 1-26: {len(fscg_r26)}")
print(f"\nUnique stadiums in rounds 1-26:")
for i, s in enumerate(sorted(fscg_r26['stadium'].unique()), 1):
    count = (fscg_r26['stadium'] == s).sum()
    print(f"  {i}. {s} ({count} matches)")

print(f"\nTotal unique stadiums: {fscg_r26['stadium'].nunique()}")

# Which teams play at each stadium?
print("\n=== STADIUM → HOME TEAM MAPPING ===")
for stadium in sorted(fscg_r26['stadium'].unique()):
    teams = fscg_r26[fscg_r26['stadium'] == stadium]['home_team'].unique()
    count = len(fscg_r26[fscg_r26['stadium'] == stadium])
    print(f"\n{stadium} ({count} home matches):")
    for t in sorted(teams):
        team_count = len(fscg_r26[(fscg_r26['stadium'] == stadium) & (fscg_r26['home_team'] == t)])
        print(f"  - {t} ({team_count}x)")

# Check if any team uses multiple stadiums
print("\n=== TEAM → STADIUMS USED (home) ===")
for team in sorted(fscg_r26['home_team'].unique()):
    stadiums = fscg_r26[fscg_r26['home_team'] == team]['stadium'].unique()
    if len(stadiums) > 1:
        print(f"⚠️  {team}: {', '.join(stadiums)}")
    else:
        print(f"   {team}: {stadiums[0]}")

Matches in rounds 1-26: 130

Unique stadiums in rounds 1-26:
  1. Arena Besa (13 matches)
  2. DG arena (9 matches)
  3. Gradski stadion Berane (15 matches)
  4. Gradski stadion Bijelo Polje (11 matches)
  5. Gradski stadion Nikšić (14 matches)
  6. Gradski stadion Pljevlja (1 matches)
  7. Gradski stadion Podgorica (12 matches)
  8. SRC Topolica (12 matches)
  9. Stadion FK Arsenal (14 matches)
  10. Stadion Mitar Mićo Goliš (12 matches)
  11. Stadion pod Vrmcem (13 matches)
  12. Trening kamp FSCG (4 matches)

Total unique stadiums: 12

=== STADIUM → HOME TEAM MAPPING ===

Arena Besa (13 home matches):
  - Dečić (13x)

DG arena (9 home matches):
  - OFK Mladost Lob.bet (9x)

Gradski stadion Berane (15 home matches):
  - Jedinstvo Franca (1x)
  - Jezero (14x)

Gradski stadion Bijelo Polje (11 home matches):
  - Jedinstvo Franca (11x)

Gradski stadion Nikšić (14 home matches):
  - Sutjeska (14x)

Gradski stadion Pljevlja (1 home matches):
  - OFK Mladost Lob.bet (1x)

Gradski stadion P

In [ ]:
# Time of day distribution from FSCG data (rounds 1-26)
fscg_r26['hour'] = fscg_r26['time'].str.extract(r'(\d+)').astype(int)

print("Kickoff time value counts (rounds 1-26):")
print(fscg_r26['hour'].value_counts().sort_index().to_string())
print(f"\nTotal matches: {len(fscg_r26)}")

Kickoff time value counts (rounds 1-26):
hour
13     8
14    31
15    21
16    13
17     8
18    13
19     8
20    25
21     3

Total matches: 130


In [ ]:
# Detail on teams with multiple home stadiums — only non-primary venues
for team in sorted(fscg_r26['home_team'].unique()):
    stadiums = fscg_r26[fscg_r26['home_team'] == team]['stadium'].value_counts()
    if len(stadiums) > 1:
        primary = stadiums.index[0]
        print(f"\n⚠️ {team} — primary: {primary} ({stadiums.iloc[0]}x), also used:")
        for s, c in stadiums.iloc[1:].items():
            rounds_used = fscg_r26[(fscg_r26['home_team'] == team) & (fscg_r26['stadium'] == s)]['round'].tolist()
            print(f"   → {s} ({c}x) in rounds: {rounds_used}")


⚠️ Jedinstvo Franca — primary: Gradski stadion Bijelo Polje (11x), also used:
   → Gradski stadion Berane (1x) in rounds: [2]

⚠️ OFK Mladost Lob.bet — primary: DG arena (9x), also used:
   → Trening kamp FSCG (2x) in rounds: [21, 25]
   → Gradski stadion Pljevlja (1x) in rounds: [23]

⚠️ Petrovac — primary: Stadion Mitar Mićo Goliš (12x), also used:
   → Trening kamp FSCG (2x) in rounds: [16, 19]


In [ ]:
# Check: which Jedinstvo home matches had "Gradski stadion" vs "Gradski stadion Berane" on FSCG
# Need to re-parse from original (before our fix) to see the raw data
with open('outputs/fscg_page.html', 'r', encoding='utf-8') as f:
    soup2 = BeautifulSoup(f.read(), 'html.parser')
tables2 = soup2.find_all('table', class_='fixtures')
t2 = tables2[1]

cur_round = None
jed_matches = []
for row in t2.find_all('tr'):
    h3 = row.find('span', class_='h3')
    if h3:
        m = re.search(r'(\d+)\.\s*kolo', h3.get_text(strip=True))
        if m: cur_round = int(m.group(1))
        continue
    if cur_round is None or cur_round > 26: continue
    tds = row.find_all('td')
    if len(tds) < 4: continue
    home_link = tds[2].find('a')
    home = home_link.get_text(strip=True) if home_link else ''
    if 'Jedinstvo' in home:
        date_div = tds[0].find('div', class_='date')
        away_link = tds[3].find('a')
        jed_matches.append({
            'round': cur_round,
            'date': date_div.get_text(strip=True) if date_div else '',
            'raw_stadium': tds[1].get_text(strip=True),
            'away': away_link.get_text(strip=True) if away_link else ''
        })

print("Jedinstvo Franca HOME matches (raw FSCG stadium names):")
for m in jed_matches:
    print(f"  R{m['round']:02d} {m['date']} | {m['raw_stadium']} | vs {m['away']}")

Jedinstvo Franca HOME matches (raw FSCG stadium names):
  R02 10.08.2025. | Gradski stadion Berane | vs Dečić
  R05 30.08.2025. | Gradski stadion | vs Mornar
  R07 17.09.2025. | Gradski stadion | vs Petrovac
  R09 27.09.2025. | Gradski stadion | vs Sutjeska
  R10 01.10.2025. | Gradski stadion | vs Arsenal
  R12 18.10.2025. | Gradski stadion | vs OFK Mladost Lob.bet
  R13 26.10.2025. | Gradski stadion | vs Bokelj sbbet
  R15 09.11.2025. | Gradski stadion | vs Budućnost
  R17 30.11.2025. | Gradski stadion | vs Jezero
  R20 21.02.2026. | Gradski stadion | vs Dečić
  R23 07.03.2026. | Gradski stadion | vs Mornar
  R25 15.03.2026. | Gradski stadion | vs Petrovac


In [9]:
# Map teams to their home stadiums — corrected with domain knowledge from FSCG
# Jezero plays in Berane, Jedinstvo plays in Bijelo Polje (except R02 at Berane)
STADIUM_MAP = {
    'FK Arsenal Tivat': 'Stadion FK Arsenal (Tivat)',
    'FK Jedinstvo Bijelo Polje': 'Gradski stadion Bijelo Polje',
    'OFK Petrovac': 'Stadion Mitar Mićo Goliš (Petrovac)',
    'FK Bokelj Kotor': 'Stadion pod Vrmcem (Kotor)',
    'FK Dečić Tuzi': 'Arena Besa (Tuzi)',
    'Mladost DG': 'DG arena (Danilovgrad)',
    'FK Jezero': 'Gradski stadion Berane',
    'FK Budućnost Podgorica': 'Gradski stadion Podgorica',
    'FK Sutjeska Nikšić': 'Gradski stadion Nikšić',
    'FK Mornar Bar': 'SRC Topolica (Bar)',
}

SHORT_NAME = {
    'FK Arsenal Tivat': 'Arsenal',
    'FK Jedinstvo Bijelo Polje': 'Jedinstvo',
    'OFK Petrovac': 'Petrovac',
    'FK Bokelj Kotor': 'Bokelj',
    'FK Dečić Tuzi': 'Dečić',
    'Mladost DG': 'Mladost DG',
    'FK Jezero': 'Jezero',
    'FK Budućnost Podgorica': 'Budućnost',
    'FK Sutjeska Nikšić': 'Sutjeska',
    'FK Mornar Bar': 'Mornar',
}

df['stadium'] = df['homeTeam_name'].map(STADIUM_MAP)
df['home_short'] = df['homeTeam_name'].map(SHORT_NAME)
df['away_short'] = df['awayTeam_name'].map(SHORT_NAME)
df['match_label'] = df['home_short'] + ' - ' + df['away_short']

# Special cases where stadium differs from default (from FSCG data)
# R23 Mladost DG vs Budućnost → Gradski stadion Pljevlja
df.loc[(df['round'] == 23) & (df['home_short'] == 'Mladost DG') & (df['away_short'] == 'Budućnost'), 'stadium'] = 'Gradski stadion Pljevlja'
# R02 Jedinstvo vs Dečić → played at Gradski stadion Berane (not Bijelo Polje)
df.loc[(df['round'] == 2) & (df['home_short'] == 'Jedinstvo') & (df['away_short'] == 'Dečić'), 'stadium'] = 'Gradski stadion Berane'
# Trening kamp FSCG — Petrovac R16, R19 and Mladost DG R21, R25
df.loc[(df['round'] == 16) & (df['home_short'] == 'Petrovac'), 'stadium'] = 'Trening kamp FSCG'
df.loc[(df['round'] == 19) & (df['home_short'] == 'Petrovac'), 'stadium'] = 'Trening kamp FSCG'
df.loc[(df['round'] == 21) & (df['home_short'] == 'Mladost DG'), 'stadium'] = 'Trening kamp FSCG'
df.loc[(df['round'] == 25) & (df['home_short'] == 'Mladost DG'), 'stadium'] = 'Trening kamp FSCG'

# Fix kickoff hours: sofascore stores UTC, FSCG has local Montenegro time (CET/CEST)
# Comparing: sofascore 12→FSCG 14, sofascore 18→FSCG 20 → consistent +2h offset
# Use local Montenegro time for accurate time-of-day analysis
df['hour_local'] = df['hour'] + 2

# Time of day classification using local time — split evening from night for lighting diversity
def time_category(hour):
    if hour < 16:
        return 'Afternoon'
    elif hour < 20:
        return 'Evening'
    else:
        return 'Night'

df['time_of_day'] = df['hour_local'].apply(time_category)
print('Matches by time of day (local Montenegro time):')
print(df['time_of_day'].value_counts())
print(f'\nKickoff hours (local):')
print(df['hour_local'].value_counts().sort_index())
print(f'\nUnique stadiums in sofascore data: {df["stadium"].nunique()}')
for s in sorted(df['stadium'].unique()):
    print(f'  {s}')

Matches by time of day (local Montenegro time):
time_of_day
Evening      60
Afternoon    42
Night        28
Name: count, dtype: int64

Kickoff hours (local):
hour_local
14     8
15    34
16    23
17    13
18    14
19    10
20    25
21     3
Name: count, dtype: int64

Unique stadiums in sofascore data: 12
  Arena Besa (Tuzi)
  DG arena (Danilovgrad)
  Gradski stadion Berane
  Gradski stadion Bijelo Polje
  Gradski stadion Nikšić
  Gradski stadion Pljevlja
  Gradski stadion Podgorica
  SRC Topolica (Bar)
  Stadion FK Arsenal (Tivat)
  Stadion Mitar Mićo Goliš (Petrovac)
  Stadion pod Vrmcem (Kotor)
  Trening kamp FSCG


In [10]:
# Mark already annotated matches
# From the video folder screenshot:
# 1. Arsenal - Dečić, 21.03.2026 (R26)
# 2. Bokelj - Jedinstvo, 01.03.2026 (R22)
# 3. Budućnost - Sutjeska, 21.02.2026 (R20)
# 4. Jezero - Jedinstvo, 21.03.2026 (R26)
# 5. Petrovac - Mornar, 21.03.2026 (R26)
# 6. Sutjeska - Mladost, ? (need to check which round)

annotated_conditions = [
    (df['home_short'] == 'Arsenal') & (df['away_short'] == 'Dečić') & (df['match_date'] == '2026-03-21'),
    (df['home_short'] == 'Bokelj') & (df['away_short'] == 'Jedinstvo') & (df['match_date'] == '2026-03-01'),
    (df['home_short'] == 'Budućnost') & (df['away_short'] == 'Sutjeska') & (df['match_date'] == '2026-02-21'),
    (df['home_short'] == 'Jezero') & (df['away_short'] == 'Jedinstvo') & (df['match_date'] == '2026-03-21'),
    (df['home_short'] == 'Petrovac') & (df['away_short'] == 'Mornar') & (df['match_date'] == '2026-03-21'),
]

# For Sutjeska - Mladost, find all possibilities
sut_mla = df[(df['home_short'] == 'Sutjeska') & (df['away_short'] == 'Mladost DG')]
print('Sutjeska - Mladost DG matches (to identify which one is annotated):')
print(sut_mla[['match_id', 'round', 'match_date', 'match_datetime', 'match_label', 'stadium']].to_string())
print()

Sutjeska - Mladost DG matches (to identify which one is annotated):
     match_id  round match_date      match_datetime            match_label                 stadium
37   14147327      8 2025-09-21 2025-09-21 16:00:00  Sutjeska - Mladost DG  Gradski stadion Nikšić
125  15281003     26 2026-03-21 2026-03-21 13:00:00  Sutjeska - Mladost DG  Gradski stadion Nikšić



In [11]:
# Add Sutjeska-Mladost — update the date once you confirm which one.
# For now assume the most recent one: R26 (2026-03-21)
annotated_conditions.append(
    (df['home_short'] == 'Sutjeska') & (df['away_short'] == 'Mladost DG') & (df['match_date'] == '2026-03-21')
)

annotated_mask = pd.concat([c for c in annotated_conditions], axis=1).any(axis=1)
df['annotated'] = annotated_mask

annotated_df = df[df['annotated']]
print(f'Annotated matches found: {len(annotated_df)}')
print()
print(annotated_df[['match_id', 'round', 'match_date', 'hour', 'match_label', 'stadium', 'time_of_day']].to_string(index=False))

Annotated matches found: 6

 match_id  round match_date  hour           match_label                             stadium time_of_day
 15280975     20 2026-02-21    13  Budućnost - Sutjeska           Gradski stadion Podgorica   Afternoon
 15280985     22 2026-03-01    13    Bokelj - Jedinstvo          Stadion pod Vrmcem (Kotor)   Afternoon
 15281003     26 2026-03-21    13 Sutjeska - Mladost DG              Gradski stadion Nikšić   Afternoon
 15281007     26 2026-03-21    13    Jezero - Jedinstvo              Gradski stadion Berane   Afternoon
 15281004     26 2026-03-21    13       Arsenal - Dečić          Stadion FK Arsenal (Tivat)   Afternoon
 15281006     26 2026-03-21    14     Petrovac - Mornar Stadion Mitar Mićo Goliš (Petrovac)     Evening


In [12]:
# Coverage analysis of already annotated matches
print('=== COVERAGE OF ANNOTATED MATCHES ===')
print()

# Stadiums covered
covered_stadiums = set(annotated_df['stadium'])
all_stadiums = set(df['stadium'])
missing_stadiums = all_stadiums - covered_stadiums
print(f'Stadiums covered: {len(covered_stadiums)}/{len(all_stadiums)}')
for s in sorted(covered_stadiums):
    print(f'  ✓ {s}')
print(f'\nMissing stadiums:')
for s in sorted(missing_stadiums):
    print(f'  ✗ {s}')

# Teams covered (as home or away in annotated matches)
print()
covered_teams = set(annotated_df['home_short']) | set(annotated_df['away_short'])
all_teams = set(df['home_short']) | set(df['away_short'])
missing_teams = all_teams - covered_teams
print(f'Teams appearing in annotated matches: {len(covered_teams)}/{len(all_teams)}')
for t in sorted(covered_teams):
    print(f'  ✓ {t}')
if missing_teams:
    print(f'\nTeams NOT in any annotated match:')
    for t in sorted(missing_teams):
        print(f'  ✗ {t}')

# Time of day
print()
print('Time of day in annotated matches:')
print(annotated_df['time_of_day'].value_counts().to_string())

=== COVERAGE OF ANNOTATED MATCHES ===

Stadiums covered: 6/12
  ✓ Gradski stadion Berane
  ✓ Gradski stadion Nikšić
  ✓ Gradski stadion Podgorica
  ✓ Stadion FK Arsenal (Tivat)
  ✓ Stadion Mitar Mićo Goliš (Petrovac)
  ✓ Stadion pod Vrmcem (Kotor)

Missing stadiums:
  ✗ Arena Besa (Tuzi)
  ✗ DG arena (Danilovgrad)
  ✗ Gradski stadion Bijelo Polje
  ✗ Gradski stadion Pljevlja
  ✗ SRC Topolica (Bar)
  ✗ Trening kamp FSCG

Teams appearing in annotated matches: 10/10
  ✓ Arsenal
  ✓ Bokelj
  ✓ Budućnost
  ✓ Dečić
  ✓ Jedinstvo
  ✓ Jezero
  ✓ Mladost DG
  ✓ Mornar
  ✓ Petrovac
  ✓ Sutjeska

Time of day in annotated matches:
time_of_day
Afternoon    5
Evening      1


In [ ]:
# Full match overview table — all matches with stadium, time, teams
cols = ['match_id', 'round', 'match_date', 'hour', 'match_label', 'stadium', 'time_of_day', 'annotated']
display_df = df[cols].sort_values(['match_date', 'hour']).reset_index(drop=True)
display_df

,match_id,round,match_date,hour,match_label,stadium,time_of_day,annotated
0,14147310,1,2025-08-04,18,Arsenal - Jedinstvo,Stadion FK Arsenal (Tivat),Night,False
1,14147408,1,2025-08-04,18,Petrovac - Bokelj,Stadion Mitar Mićo Goliš (Petrovac),Night,False
2,14147300,1,2025-08-04,18,Dečić - Mladost DG,Arena Besa (Tuzi),Night,False
3,14147319,1,2025-08-04,18,Jezero - Budućnost,Gradski stadion Berane,Night,False
4,14147314,1,2025-08-04,19,Sutjeska - Mornar,Gradski stadion Nikšić,Night,False
...,...,...,...,...,...,...,...,...
125,15281003,26,2026-03-21,13,Sutjeska - Mladost DG,Gradski stadion Nikšić,Afternoon,True
126,15281005,26,2026-03-21,13,Bokelj - Budućnost,Stadion pod Vrmcem (Kotor),Afternoon,False
127,15281007,26,2026-03-21,13,Jezero - Jedinstvo,Gradski stadion Berane,Afternoon,True
128,15281004,26,2026-03-21,13,Arsenal - Dečić,Stadion FK Arsenal (Tivat),Afternoon,True


In [13]:
# === RECOMMENDED MATCHES TO ANNOTATE ===
# Strategy: 
# 1. Must cover all missing stadiums (including Trening kamp FSCG)
# 2. Every team should appear, with uniform distribution (avoid >3 per team)
# 3. Max 2 matches per stadium across all 16 (6 done + 10 new)
# 4. Mix of time slots — include night matches (20:00+)
# 5. Spread across different months/rounds

not_annotated = df[~df['annotated']].copy()

def score_match(row, selected_ids=None):
    """Score a match by how much annotation diversity it would add."""
    if selected_ids is None:
        selected_ids = set()
    
    already = df[df['annotated'] | df['match_id'].isin(selected_ids)]
    
    score = 0
    
    # Stadium not yet covered → big bonus
    covered_stad = already['stadium'].value_counts()
    stad_count = covered_stad.get(row['stadium'], 0)
    if stad_count == 0:
        score += 15  # new stadium — highest priority
    elif stad_count == 1:
        score += 2   # second match at this stadium is OK
    else:
        score -= 20  # penalize 3rd+ match at same stadium
    
    # Teams not yet appearing → bonus per new team
    covered_t = set(already['home_short']) | set(already['away_short'])
    if row['home_short'] not in covered_t:
        score += 5
    if row['away_short'] not in covered_t:
        score += 5
    
    # Team balance — penalize teams that already appear too often
    from collections import Counter
    team_counts = Counter()
    for _, r in already.iterrows():
        team_counts[r['home_short']] += 1
        team_counts[r['away_short']] += 1
    home_count = team_counts.get(row['home_short'], 0)
    away_count = team_counts.get(row['away_short'], 0)
    # Prefer matches with underrepresented teams
    if home_count <= 1:
        score += 3
    elif home_count >= 4:
        score -= 3
    if away_count <= 1:
        score += 3
    elif away_count >= 4:
        score -= 3
    
    # Time of day not yet well represented
    time_counts = already['time_of_day'].value_counts()
    if row['time_of_day'] not in time_counts.index or time_counts.get(row['time_of_day'], 0) < 2:
        score += 3
    
    # Night match bonus (20:00+ local) — floodlit conditions are very different from daylight
    if row['hour_local'] >= 20:
        night_count = (already['hour_local'] >= 20).sum()
        if night_count == 0:
            score += 8
        elif night_count < 3:
            score += 4
    
    # Month diversity
    covered_months = set(already['month'])
    if row['month'] not in covered_months:
        score += 3
    
    # Round diversity — prefer different rounds
    covered_rounds = set(already['round'])
    if row['round'] not in covered_rounds:
        score += 1
    
    return score


# Greedy selection: pick highest-scoring match, update coverage, repeat
selected_ids = set()
NUM_TO_SELECT = 10

for i in range(NUM_TO_SELECT):
    candidates = not_annotated[~not_annotated['match_id'].isin(selected_ids)].copy()
    candidates['score'] = candidates.apply(lambda row: score_match(row, selected_ids), axis=1)
    best = candidates.sort_values('score', ascending=False).iloc[0]
    selected_ids.add(best['match_id'])
    print(f'{i+1}. [Score {best["score"]:.0f}] R{best["round"]:02.0f} {best["match_date"].strftime("%Y-%m-%d")} {best["hour_local"]:02.0f}:00 | {best["match_label"]} | {best["stadium"]} | {best["time_of_day"]}')

print(f'\nSelected {len(selected_ids)} matches.')

1. [Score 36] R01 2025-08-04 20:00 | Dečić - Mladost DG | Arena Besa (Tuzi) | Night
2. [Score 32] R07 2025-09-17 20:00 | Mornar - Budućnost | SRC Topolica (Bar) | Night
3. [Score 25] R10 2025-10-01 17:00 | Jedinstvo - Arsenal | Gradski stadion Bijelo Polje | Evening
4. [Score 25] R19 2025-12-10 14:00 | Petrovac - Bokelj | Trening kamp FSCG | Afternoon
5. [Score 20] R05 2025-08-31 20:00 | Mladost DG - Budućnost | DG arena (Danilovgrad) | Night
6. [Score 16] R23 2026-03-07 15:00 | Mladost DG - Budućnost | Gradski stadion Pljevlja | Afternoon
7. [Score 9] R14 2025-11-02 16:00 | Jezero - Arsenal | Gradski stadion Berane | Evening
8. [Score 3] R02 2025-08-10 20:00 | Mornar - Arsenal | SRC Topolica (Bar) | Night
9. [Score 3] R03 2025-08-17 21:00 | Sutjeska - Petrovac | Gradski stadion Nikšić | Night
10. [Score 3] R04 2025-08-24 19:00 | Bokelj - Jedinstvo | Stadion pod Vrmcem (Kotor) | Evening

Selected 10 matches.


In [14]:
# Final coverage check after adding recommended matches
selected_df = df[df['match_id'].isin(selected_ids)]
combined = pd.concat([annotated_df, selected_df])

print('=== COVERAGE AFTER ADDING RECOMMENDED MATCHES ===')
print(f'Total annotated matches: {len(combined)} ({len(annotated_df)} existing + {len(selected_df)} new)')
print(f'Total frames (est.): ~{len(combined) * 60}')
print()

# Stadiums
print(f'Stadiums: {combined["stadium"].nunique()}/{df["stadium"].nunique()}')
for s in sorted(combined['stadium'].unique()):
    count = (combined['stadium'] == s).sum()
    print(f'  ✓ {s} ({count} match{"es" if count > 1 else ""})')

# Teams  
print()
all_appearing = set(combined['home_short']) | set(combined['away_short'])
print(f'Teams: {len(all_appearing)}/{len(all_teams)}')
# Count appearances per team
from collections import Counter
team_counts = Counter()
for _, row in combined.iterrows():
    team_counts[row['home_short']] += 1
    team_counts[row['away_short']] += 1
for t in sorted(team_counts.keys()):
    print(f'  {t}: {team_counts[t]} appearances')

# Time distribution
print()
print('Time of day distribution:')
print(combined['time_of_day'].value_counts().to_string())

# Month distribution
print()
print('Month distribution:')
print(combined['month'].value_counts().sort_index().to_string())

=== COVERAGE AFTER ADDING RECOMMENDED MATCHES ===
Total annotated matches: 16 (6 existing + 10 new)
Total frames (est.): ~960

Stadiums: 12/12
  ✓ Arena Besa (Tuzi) (1 match)
  ✓ DG arena (Danilovgrad) (1 match)
  ✓ Gradski stadion Berane (2 matches)
  ✓ Gradski stadion Bijelo Polje (1 match)
  ✓ Gradski stadion Nikšić (2 matches)
  ✓ Gradski stadion Pljevlja (1 match)
  ✓ Gradski stadion Podgorica (1 match)
  ✓ SRC Topolica (Bar) (2 matches)
  ✓ Stadion FK Arsenal (Tivat) (1 match)
  ✓ Stadion Mitar Mićo Goliš (Petrovac) (1 match)
  ✓ Stadion pod Vrmcem (Kotor) (2 matches)
  ✓ Trening kamp FSCG (1 match)

Teams: 10/10
  Arsenal: 4 appearances
  Bokelj: 3 appearances
  Budućnost: 4 appearances
  Dečić: 2 appearances
  Jedinstvo: 4 appearances
  Jezero: 2 appearances
  Mladost DG: 4 appearances
  Mornar: 3 appearances
  Petrovac: 3 appearances
  Sutjeska: 3 appearances

Time of day distribution:
time_of_day
Afternoon    7
Night        5
Evening      4

Month distribution:
month
2     1


In [15]:
# Summary table of ALL matches to annotate (existing + recommended)
combined_display = combined[['match_id', 'round', 'match_date', 'hour_local', 'match_label', 'stadium', 'time_of_day', 'annotated']].copy()
combined_display['status'] = combined_display['annotated'].map({True: '✓ DONE', False: '→ TO DO'})
combined_display = combined_display.sort_values('match_date').reset_index(drop=True)
print(combined_display[['round', 'match_date', 'hour_local', 'match_label', 'stadium', 'time_of_day', 'status']].to_string(index=False))

 round match_date  hour_local            match_label                             stadium time_of_day  status
     1 2025-08-04          20     Dečić - Mladost DG                   Arena Besa (Tuzi)       Night → TO DO
     2 2025-08-10          20       Mornar - Arsenal                  SRC Topolica (Bar)       Night → TO DO
     3 2025-08-17          21    Sutjeska - Petrovac              Gradski stadion Nikšić       Night → TO DO
     4 2025-08-24          19     Bokelj - Jedinstvo          Stadion pod Vrmcem (Kotor)     Evening → TO DO
     5 2025-08-31          20 Mladost DG - Budućnost              DG arena (Danilovgrad)       Night → TO DO
     7 2025-09-17          20     Mornar - Budućnost                  SRC Topolica (Bar)       Night → TO DO
    10 2025-10-01          17    Jedinstvo - Arsenal        Gradski stadion Bijelo Polje     Evening → TO DO
    14 2025-11-02          16       Jezero - Arsenal              Gradski stadion Berane     Evening → TO DO
    19 2025-12-10  